# 02 · Comparación de representaciones

**Taller de una hora.** Tres formas de convertir una reseña en un vector —**bolsa de palabras**,
**TF-IDF** y **embeddings**— aplicadas al mismo corpus y evaluadas sobre la misma tarea.

> Nota metodológica: las tres representaciones usan el texto en minúsculas, pero la línea base de embeddings excluye stopwords de spaCy, mientras que BoW y TF-IDF conservan todos los tokens que reconoce su tokenizador. Esta diferencia se mantiene visible porque forma parte de la decisión de preprocesamiento; no debe presentarse como una comparación con idéntica entrada.

### La tarea

Dada una reseña, **recuperar las más parecidas del corpus**. Es la tarea que subyace a buscar
quejas del mismo tipo, agrupar incidencias por causa o encontrar menciones equivalentes de un
producto.

Una representación no es mejor por ser más moderna: es mejor si **ordena mejor los resultados
de la tarea que os interesa**. Al terminar tenéis que haber anotado, para cada una, un acierto
y un fallo sobre vuestro corpus.

---

> El corpus es el mismo del taller anterior: reseñas de producto de Amazon en español,
> `mteb/amazon_reviews_multi`. Uso docente; se descarga a `data/raw/`, que no se versiona.

## 0 · Preparación

Ejecutad esta celda nada más abrir el notebook. El modelo de vectores tarda un par de
minutos la primera vez.

In [ ]:
%pip install -q scikit-learn spacy pandas pyarrow

import importlib.util, subprocess, sys

# es_core_news_sm NO tiene vectores de palabra: hace falta el modelo mediano.
if importlib.util.find_spec('es_core_news_md') is None:
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'es_core_news_md'], check=True)

print('entorno listo')

## 1 · El corpus de trabajo

Para poder seguir los cálculos a mano, el bloque central usa **cinco reseñas** escritas a
propósito. Al final del notebook se repite todo sobre el corpus completo.

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path

resenas = [
    'el producto llegó roto y el embalaje venía abierto',        # R0
    'el artículo vino estropeado dentro de una caja abierta',    # R1
    'producto perfecto y entrega muy rápida',                    # R2
    'entrega puntual, el producto es perfecto',                  # R3
    'el envío tardó tres semanas en llegar',                     # R4
]
for i, r in enumerate(resenas):
    print(f'R{i}: {r}')

print()
print('CONSULTA: R0. ¿Cuál es la reseña más parecida?')
print('A ojo: R1 dice lo mismo con otras palabras. Veamos si cada método lo detecta.')

Fijaos en el par **R0 y R1**: describen el mismo problema sin compartir casi ningún término
(*roto* / *estropeado*, *producto* / *artículo*, *llegó* / *vino*). Y en **R2 y R3**, que dicen
lo mismo compartiendo palabras. Esa diferencia es la que va a separar a los tres métodos.

## 2 · Bolsa de palabras

`CountVectorizer` construye el vocabulario y cuenta. Cada fila de la matriz es una reseña;
cada columna, un término.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer()
X = bow.fit_transform(resenas)

M = pd.DataFrame(X.toarray(), columns=bow.get_feature_names_out(),
                 index=[f'R{i}' for i in range(len(resenas))])
print(f'{M.shape[0]} documentos x {M.shape[1]} términos')
print(f'celdas a cero: {100*(M.values==0).mean():.0f}%  <- la matriz es dispersa')
M

La dimensión del vector es el tamaño del vocabulario, y la mayoría de las celdas valen cero:
cada reseña emplea una fracción mínima de los términos disponibles.

## 3 · Coseno y vecinos con bolsa de palabras

El coseno mide el ángulo entre dos vectores, de modo que la comparación no depende de la
longitud del documento.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def vecinos(matriz, consulta=0, etiqueta=''):
    """Ordena los documentos por coseno con la consulta, excluyendola."""
    sim = cosine_similarity(matriz)[consulta]
    orden = [(i, sim[i]) for i in np.argsort(-sim) if i != consulta]
    print(f'--- {etiqueta} · vecinos de R{consulta} ---')
    for i, s in orden:
        print(f'  R{i}  cos = {s:.3f}   {resenas[i][:52]}')
    return orden

orden_bow = vecinos(X.toarray(), 0, 'BoW')

**R1, que es la paráfrasis, no es la más próxima.** Comparte con R0 solo las palabras
funcionales y *abierta*; las que llevan el significado son distintas. Para la bolsa de
palabras, *roto* y *estropeado* son dos dimensiones sin ninguna relación, igual que *roto* y
*paraguas*.

Este es el límite de fondo de las representaciones por recuento: **solo reconocen coincidencia
literal de términos.**

## 4 · Ponderación TF-IDF

TF-IDF rebaja el peso de los términos presentes en muchos documentos y realza los que
aparecen en pocos.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

print('Parámetros TF-IDF: smooth_idf=', True, ', sublinear_tf=', False, ', norm=', 'l2')
print('La implementación de scikit-learn usa IDF suavizado y normalización L2; no coincide exactamente con la fórmula manual de las diapositivas.')

tfidf = TfidfVectorizer()
Xt = tfidf.fit_transform(resenas)

pesos = pd.DataFrame({'idf': tfidf.idf_}, index=tfidf.get_feature_names_out())
pesos['documentos'] = np.asarray((X > 0).sum(axis=0)).ravel()
print('Términos con MENOS peso (aparecen en más reseñas):')
print(pesos.sort_values('idf').head(6).to_string())
print()
print('Términos con MÁS peso (aparecen en una sola):')
print(pesos.sort_values('idf', ascending=False).head(6).to_string())

`el` y `producto` reciben el peso más bajo porque están en casi todas las reseñas: no sirven
para distinguirlas. Los términos que aparecen una sola vez reciben el peso máximo.

Es el mismo efecto que buscabais con la lista de stopwords del taller anterior, pero obtenido
**de los propios datos** en lugar de una lista escrita de antemano.

In [ ]:
orden_tfidf = vecinos(Xt.toarray(), 0, 'TF-IDF')

**Los valores del coseno bajan y el orden apenas cambia.** Conviene detenerse en esto: un
coseno más bajo no significa un método peor. TF-IDF modifica la geometría del espacio, y lo
que se evalúa es **el orden de los resultados**, no la magnitud de la puntuación.

Aquí TF-IDF no resuelve el problema de fondo, porque R0 y R1 siguen sin compartir los términos
que importan. Ponderar mejor no crea una relación que no existe en el recuento.

## 5 · Embeddings de palabras

Un embedding asigna a cada palabra un vector aprendido a partir de los contextos en que
aparece en un corpus grande. Palabras que aparecen en contextos parecidos reciben vectores
próximos, **aunque nunca coincidan en el mismo documento**.

In [ ]:
import spacy

nlp = spacy.load('es_core_news_md')
print('modelo de embeddings: es_core_news_md', spacy.__version__)
print('vectores disponibles:', nlp.vocab.vectors.shape[0],
      'palabras ×', nlp.vocab.vectors.shape[1], 'dimensiones')

def cos(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(a @ b / (na * nb)) if na and nb else 0.0

v = lambda p: nlp(p)[0].vector
print()
for a, b in [('roto', 'estropeado'), ('perfecto', 'excelente'),
             ('roto', 'paraguas'), ('roto', 'defectuoso')]:
    print(f'  cos({a}, {b}) = {cos(v(a), v(b)):+.3f}')

`roto` y `estropeado` quedan próximos; `roto` y `paraguas`, lejos. **Esa relación no estaba
en nuestras cinco reseñas**: procede del corpus con el que se entrenó el modelo.

Fijaos también en `roto` / `defectuoso`, que salen mucho menos próximos de lo que cabría
esperar. El modelo captura unas relaciones sí y otras no; no es un diccionario de sinónimos.

## 6 · De palabras a documentos

El modelo devuelve un vector por palabra, y necesitamos uno por reseña. La composición más
simple es la **media de los vectores de las palabras con vector disponible**.

Es una línea base, no la única opción: al promediar se pierde el orden y una negación puede
diluirse entre muchas palabras.

In [ ]:
def vector_doc(texto):
    doc = nlp(texto.lower())
    vs = [t.vector for t in doc
          if t.has_vector and not (t.is_punct or t.is_space or t.is_stop)]
    cubiertas = len(vs)
    total = sum(1 for t in doc if not (t.is_punct or t.is_space or t.is_stop))
    if not vs:
        return np.zeros(nlp.vocab.vectors.shape[1]), 0, total
    return np.mean(vs, axis=0), cubiertas, total

E, cob = [], []
for r in resenas:
    vd, c, tot = vector_doc(r)
    E.append(vd); cob.append((c, tot))
E = np.vstack(E)

print('cobertura por reseña (palabras con vector / palabras con contenido):')
for i, (c, tot) in enumerate(cob):
    print(f'  R{i}: {c}/{tot}')
print(f'\ncada reseña es ahora un vector de {E.shape[1]} dimensiones')

**La cobertura hay que medirla siempre.** Un documento cuyas palabras no estén en el modelo
queda representado por un vector de ceros, y a partir de ahí cualquier comparación es ruido.

In [ ]:
orden_emb = vecinos(E, 0, 'Embeddings (media)')

## 7 · Las tres, una al lado de la otra

In [ ]:
tabla = pd.DataFrame({
    'reseña': [resenas[i][:44] for i in range(1, len(resenas))],
    'BoW': [dict(orden_bow)[i] for i in range(1, len(resenas))],
    'TF-IDF': [dict(orden_tfidf)[i] for i in range(1, len(resenas))],
    'Embeddings': [dict(orden_emb)[i] for i in range(1, len(resenas))],
}, index=[f'R{i}' for i in range(1, len(resenas))]).round(3)
print('Similitud con R0 = "el producto llegó roto y el embalaje venía abierto"\n')
print(tabla.to_string())

print('\nprimer vecino según cada método:')
for nom, orden in [('BoW', orden_bow), ('TF-IDF', orden_tfidf), ('Embeddings', orden_emb)]:
    print(f'  {nom:12} -> R{orden[0][0]}')

Los métodos por recuento sitúan primero a la reseña que **comparte palabras**; los embeddings,
a la que **comparte significado**. Para la tarea planteada —encontrar reseñas que describen el
mismo problema— la segunda respuesta es la correcta.

Eso no convierte a los embeddings en la opción por defecto, como muestra la sección siguiente.

## 8 · El fallo que hay que conocer

Los embeddings sitúan próximas las palabras que aparecen en **contextos parecidos**. Los
antónimos aparecen exactamente en los mismos contextos.

In [ ]:
print('Pares de significado OPUESTO:')
for a, b in [('caro', 'barato'), ('rápido', 'lento'), ('bueno', 'malo')]:
    print(f'  cos({a}, {b}) = {cos(v(a), v(b)):+.3f}')

print()
a = 'la entrega fue muy rápida'
b = 'la entrega fue muy lenta'
va, _, _ = vector_doc(a)
vb, _, _ = vector_doc(b)
print(f'{a!r}\n{b!r}\n  cos = {cos(va, vb):+.3f}')

`caro` y `barato` quedan casi pegados, y dos reseñas de sentimiento opuesto resultan muy
similares. **No es un error del modelo**: mide relación temática, y ambas hablan de lo mismo.

La consecuencia es concreta para vuestros proyectos: una representación que agrupa bien por
tema puede ser **inservible para clasificar sentimiento**. Es el motivo por el que la elección
de representación depende de la tarea y hay que justificarla con resultados, no con el
prestigio del método.

## 9 · Sobre el corpus real

Se repite el procedimiento sobre las reseñas de Amazon. Aquí las tres representaciones se
construyen sobre miles de documentos.

In [ ]:
URL = ('https://huggingface.co/api/datasets/mteb/amazon_reviews_multi/parquet/es/test/0.parquet')
raiz = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
cache = raiz / 'data' / 'raw' / 'resenas_es.parquet'
cache.parent.mkdir(parents=True, exist_ok=True)
df = pd.read_parquet(cache) if cache.exists() else pd.read_parquet(URL)
if not cache.exists():
    df.to_parquet(cache)

N = 800          # subconjunto, para que el taller sea ágil
corpus = df['text'].astype(str).str.replace('\n\n', '. ', regex=False).head(N).tolist()
print(f'{len(corpus)} reseñas')
print('consulta:', corpus[0][:100])

In [ ]:
Xb = CountVectorizer().fit_transform(corpus)
Xt2 = TfidfVectorizer().fit_transform(corpus)
print(f'vocabulario: {Xb.shape[1]} términos')

docs = list(nlp.pipe([c.lower() for c in corpus], batch_size=100))
Em, sin_vector = [], 0
for d in docs:
    vs = [t.vector for t in d if t.has_vector and not (t.is_punct or t.is_space or t.is_stop)]
    if not vs:
        sin_vector += 1
        vs = [np.zeros(nlp.vocab.vectors.shape[1])]
    Em.append(np.mean(vs, axis=0))
Em = np.vstack(Em)
print(f'documentos sin ninguna palabra con vector: {sin_vector}')

In [ ]:
def top3(matriz, nombre, consulta=0):
    sim = cosine_similarity(matriz[consulta].reshape(1, -1), matriz)[0]
    print(f'--- {nombre} ---')
    for i in np.argsort(-sim)[1:4]:
        print(f'  cos {sim[i]:.3f}  {corpus[i][:88]}')
    print()

print('CONSULTA:', corpus[0][:110], '\n')
top3(Xb.toarray(), 'BoW')
top3(Xt2.toarray(), 'TF-IDF')
top3(Em, 'Embeddings (media)')

**Leed los resultados, no solo los números.** Para cada método, decidid si las tres reseñas
recuperadas responden de verdad a la consulta.

Y comparad esta sección con la 7, porque **el resultado se invierte**. En las cinco reseñas de
juguete ganaban los embeddings; sobre el corpus real, para esta consulta, quienes recuperan la
misma queja son la bolsa de palabras y TF-IDF, mientras los embeddings devuelven incidencias
del mismo ámbito pero de otro tipo.

La explicación es que aquí la consulta y su respuesta **sí comparten las palabras que
importan**, y en ese caso la coincidencia literal es una señal excelente. La ventaja de los
embeddings aparece cuando el vocabulario difiere, no siempre.

De ahí la conclusión operativa del taller: **con un solo ejemplo no se distingue un método
bueno de uno afortunado.** Cambiad el índice de la consulta y repetid con dos o tres reseñas
más antes de anotar nada.

## 10 · Registro

Rellenad esta tabla con **vuestras** observaciones y llevadla a `docs/bitacora.md`.

In [ ]:
print('| Representación | Primer vecino | ¿Responde a la consulta? | Acierto observado | Fallo observado |')
print('|---|---|---|---|---|')
for nombre in ['BoW', 'TF-IDF', 'Embeddings']:
    print(f'| {nombre} | RELLENAD | RELLENAD | RELLENAD | RELLENAD |')
print()
print('Representación elegida provisionalmente: RELLENAD')
print('Por qué, en términos de la tarea:        RELLENAD')
print('Qué falta comprobar antes del Hito 2:    RELLENAD')

---

## Antes de cerrar

- [ ] Habéis ejecutado la comparación con **al menos tres consultas distintas**.
- [ ] Para cada representación hay **un acierto y un fallo** anotados, con la reseña concreta.
- [ ] La tabla está en `docs/bitacora.md` con la elección provisional y su motivo.
- [ ] El notebook está confirmado en el repositorio.

La elección **no es definitiva**: en el Hito 1 se documenta la exploración, y la representación
junto con el modelo base se defienden en el Hito 2.

## Con vuestro corpus

Sustituid la lista `corpus` de la sección 9 por vuestros documentos. El resto funciona igual.
Comprobad dos cosas antes de sacar conclusiones: la **cobertura** del modelo de vectores sobre
vuestro vocabulario, y si vuestra tarea se parece más a agrupar por tema o a distinguir
sentimiento, porque de eso depende cuál de las tres os convenga.